# M11.10 · Modelli RL ve planlama — Dyna, öğrenilmiş modelle MPC, Monte Carlo ağaç araması

Geçiş ve ödül modelini açıkça öğren, modelle planla: örnek verimliliği artar, karmaşıklık da. Model öğrenme (sayımlar, doğrusal ve sinir ağı modelleri), kısmi planlama (her gerçek adımda birkaç simüle Q-öğrenme adımı: Dyna), tekrar tamponu ile modelin farkı, değişen ortamda Dyna-Q+, öncelikli tarama; şimdiki durumdan planlama: model öngörülü kontrol (MPC) ve Monte Carlo ağaç araması (seçim, genişletme, simülasyon, geri yayılım; UCT ve AlphaGo'nun PUCT'u). Saf Python'da Dyna labirenti, NumPy'da kesinlik eşdeğerli planlama, PyTorch'ta öğrenilmiş sinir ağı modeliyle CartPole MPC'si ve vektörel rastgele oyunlarla MCTS.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-10-planning.html](https://m-gonul.github.io/pytorch-lab/m11-10-planning.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_10_planning.py
CHECKS = {
 "b": "from labcheck import no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def env():\n        f = need(ns, \"maze_step\")\n        ensure(f((2, 0), 2) == ((2, 0), 0.0, False) and f((2, 1), 3) == ((2, 1), 0.0, False), \"kenar ve duvar\")\n        ensure(f((1, 8), 0) == ((0, 8), 1.0, True), \"hedef\")\n    R_.test(\"maze_step\", env)\n\n    def dq():\n        rows, res = [], {}\n        for n in (0, 5, 50):\n            S = [need(ns, \"dyna_q\")(50, n, s)[1] for s in range(10)]\n            res[n] = [sum(x[k] for x in S) / 10 for k in range(50)]\n            rows.append([n] + [f\"{res[n][k]:.0f}\" for k in (0, 1, 2, 5, 10, 49)])\n        ensure(res[50][1] < res[5][1] < res[0][1], \"2. bölümde planlama adımı çoğaldıkça daha kısa\")\n        ensure(res[50][5] < 25 and res[0][5] > 50, \"n = 50 birkaç bölümde optimal yola yaklaşmalı\")\n        R_.table(\"Dyna labirenti, bölüm başına adım (10 deney; en kısa yol 14)\", [\"n\", \"1\", \"2\", \"3\", \"6\", \"11\", \"50. bölüm\"], rows,\n                 note=\"İlk bölüm planlama olmadan da aynı uzunluktadır (ödül görülmemiştir). Ödül bir kez görülünce n = 50 değer bilgisini model üzerinden geriye hızla yayar (Sutton Şekil 8.2–8.3).\")\n    R_.test(\"Dyna-Q\", dq)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"fit_model\"), max_loops(src, 1, func=\"plan\"), max_loops(src, 0, func=\"policy_value\")))\n    rng = np.random.default_rng(1)\n    nS, nA, gamma = 15, 3, 0.9\n    P = rng.random((nA, nS, nS)) ** 4\n    P /= P.sum(-1, keepdims=True)\n    R = rng.random((nS, nA))\n\n    def data(n, seed):\n        r = np.random.default_rng(seed)\n        S_ = r.integers(0, nS, n)\n        A_ = r.integers(0, nA, n)\n        cdf = P[A_, S_].cumsum(1)\n        S2_ = (r.random(n)[:, None] > cdf).sum(1).clip(max=nS - 1)\n        return S_, A_, R[S_, A_], S2_\n\n    def fm():\n        S_, A_, R__, S2_ = data(200000, 0)\n        Ph, Rh = need(ns, \"fit_model\")(S_, A_, R__, S2_, nS, nA)\n        close(Ph.sum(-1), np.ones((nA, nS)), what=\"satır toplamları\")\n        ensure(np.abs(Ph - P).max() < 0.02 and np.abs(Rh - R).max() < 1e-12, \"çok veride gerçek modele yakınsar\")\n        Ph2, _ = ns[\"fit_model\"](S_[:3], A_[:3], R__[:3], S2_[:3], nS, nA)\n        ensure(np.allclose(Ph2.sum(-1), 1), \"görülmemiş çiftlerde de geçerli dağılım\")\n    R_.test(\"fit_model\", fm)\n\n    def cmp():\n        Qs = need(ns, \"plan\")(P, R, gamma)\n        Vs = need(ns, \"policy_value\")(P, R, Qs.argmax(1), gamma)\n        rows = []\n        res = {}\n        for n in (300, 1000, 3000, 10000):\n            vm = vq = vq10 = 0\n            for seed in range(5):\n                S_, A_, R__, S2_ = data(n, seed + 10)\n                Ph, Rh = ns[\"fit_model\"](S_, A_, R__, S2_, nS, nA)\n                vm += ns[\"policy_value\"](P, R, ns[\"plan\"](Ph, Rh, gamma).argmax(1), gamma).mean()\n                vq += ns[\"policy_value\"](P, R, need(ns, \"q_learning_replay\")(S_, A_, R__, S2_, nS, nA, gamma, 1).argmax(1), gamma).mean()\n                vq10 += ns[\"policy_value\"](P, R, ns[\"q_learning_replay\"](S_, A_, R__, S2_, nS, nA, gamma, 10).argmax(1), gamma).mean()\n            res[n] = (vm / 5, vq / 5, vq10 / 5)\n            rows.append([n, f\"{vm / 5:.3f}\", f\"{vq / 5:.3f}\", f\"{vq10 / 5:.3f}\"])\n        ensure(res[1000][0] > res[1000][1] and Vs.mean() - res[10000][0] < 0.05, \"modelli planlama örnek verimli\")\n        R_.table(f\"Açgözlü politikanın gerçek ortalama değeri (optimal {Vs.mean():.3f}), 5 tekrar\", [\"geçiş sayısı\", \"model + planlama\", \"Q-öğrenme (1 geçiş)\", \"Q-öğrenme (10 tekrar)\"], rows,\n                 note=\"Aynı veri: model her geçişi tüm planlama boyunca kullanır; Q-öğrenme her örnekten tek küçük adım alır. Tekrar (replay) aradaki farkı kapatır — Dyna ile tamponun yakınlığı.\")\n    R_.test(\"model tabanlı vs modelsiz\", cmp)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\nclass TrueModel(torch.nn.Module):\n    def __init__(self, step):\n        super().__init__()\n        self.step = step\n\n    def forward(self, x, a):\n        return self.step(x, a)[0]\n\n\ndef check(ns, R_):\n    g = torch.Generator().manual_seed(0)\n\n    def data():\n        X, A, X2 = need(ns, \"random_data\")(200, g)\n        ensure(X.shape[1] == 4 and X.shape[0] == A.shape[0] == X2.shape[0] > 3000, \"veri şekilleri\")\n        close(X2.numpy(), ns[\"cartpole_step\"](X, A)[0].numpy(), atol=1e-6, what=\"geçişler gerçek fizikten\")\n        m = need(ns, \"Model\")()\n        loss = need(ns, \"fit\")(m, X, A, X2, 100, 0)\n        Xt, At, X2t = ns[\"random_data\"](20, torch.Generator().manual_seed(9))\n        with torch.no_grad():\n            err = ((m(Xt, At) - X2t) / (X2t - Xt).std(0)).pow(2).mean()\n            base = ((Xt - X2t) / (X2t - Xt).std(0)).pow(2).mean()\n        ensure(float(err) < 0.01 * float(base), f\"model fark tahmininde iyi olmalı ({float(err):.2e} vs {float(base):.2e})\")\n        lens_true = [need(ns, \"run_mpc\")(TrueModel(ns[\"cartpole_step\"]), g, 200) for _ in range(2)]\n        lens = [ns[\"run_mpc\"](m, g, 200) for _ in range(3)]\n        ensure(min(lens) >= 150, f\"öğrenilmiş modelle MPC dengede tutmalı ({lens})\")\n        R_.table(\"CartPole, rastgele atış MPC (500 dizi × 20 adım)\", [\"model\", \"bölüm uzunlukları (en fazla 200)\"],\n                 [[\"gerçek fizik\", \", \".join(map(str, lens_true))], [\"öğrenilmiş ağ (yalnız rastgele veri, ~4000 geçiş)\", \", \".join(map(str, lens))]],\n                 note=f\"Rastgele politika ~22 adım dayanır. Model test hatası (normalize Δ, MSE) {float(err):.1e}. Hiç ödül sinyali olmadan toplanan veriden öğrenilen model, bilinen bir maliyetle planlanınca görevi çözer — modelsiz DQN/A2C'nin yüz binlerce adımına karşı birkaç bin geçiş.\")\n    R_.test(\"model öğrenme ve MPC\", data)\n",
 "e": "import random\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\nLN = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]\n\n\ndef term(b):\n    for l in LN:\n        s = b[l[0]] + b[l[1]] + b[l[2]]\n        if abs(s) == 3:\n            return s // 3\n    return 0 if all(b) else None\n\n\nMEMO = {}\n\n\ndef perfect(b, p):\n    k = (tuple(b), p)\n    if k in MEMO:\n        return MEMO[k]\n    t = term(b)\n    if t is not None:\n        return t, None\n    best = None\n    for m in range(9):\n        if b[m] == 0:\n            b2 = b[:]\n            b2[m] = p\n            v = perfect(b2, -p)[0]\n            if best is None or v * p > best[0] * p:\n                best = (v, m)\n    MEMO[k] = best\n    return best\n\n\ndef loop_playouts(board, player, n, rng):\n    tot = 0\n    for _ in range(n):\n        b, p = board[:], player\n        while term(b) is None:\n            m = rng.choice([i for i in range(9) if b[i] == 0])\n            b[m] = p\n            p = -p\n        tot += term(b)\n    return tot / n\n\n\ndef play(px, po, seed, ns):\n    rng = random.Random(seed)\n    g = torch.Generator().manual_seed(seed)\n    b, p = [0] * 9, 1\n    while term(b) is None:\n        f = px if p == 1 else po\n        if f == \"random\":\n            m = rng.choice([i for i in range(9) if b[i] == 0])\n        elif f == \"perfect\":\n            m = perfect(b, p)[1]\n        else:\n            m = ns[\"mcts_move\"](b, p, 200, 16, g)\n        b[m] = p\n        p = -p\n    return term(b)\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"winner döngüsüz, playout yalnız hamle döngüsü\", lambda: (max_loops(src, 0, func=\"winner\"), max_loops(src, 1, func=\"batched_playouts\")))\n\n    def w():\n        B = torch.tensor([[1, 1, 1, 0, -1, -1, 0, 0, 0], [-1, 1, 0, -1, 1, 0, -1, 0, 1], [1, -1, 1, 1, -1, -1, -1, 1, 1], [0] * 9])\n        ensure(need(ns, \"winner\")(B).tolist() == [1, -1, 0, 0], \"kazanan\")\n    R_.test(\"winner\", w)\n\n    def po():\n        g = torch.Generator().manual_seed(0)\n        r = need(ns, \"batched_playouts\")([0] * 9, 1, 100000, g)\n        ensure(abs(float((r == 1).double().mean()) - 0.585) < 0.01 and abs(float((r == -1).double().mean()) - 0.288) < 0.01, \"boş tahtada X %58.5, O %28.8\")\n        b = [1, 1, 0, -1, -1, 0, 0, 0, 0]\n        close(float(ns[\"batched_playouts\"](b, 1, 20000, g).mean()), loop_playouts(b, 1, 20000, random.Random(0)), atol=0.03, what=\"döngülü oyunlarla aynı dağılım\")\n        t1 = bench(lambda: ns[\"batched_playouts\"]([0] * 9, 1, 10000, g), repeat=3, number=1)\n        t2 = bench(lambda: loop_playouts([0] * 9, 1, 10000, random.Random(0)), repeat=1, number=1)\n        R_.table(\"10 000 rastgele oyun\", [\"yöntem\", \"süre\"], [[\"tensör (paralel)\", fmt_time(t1)], [\"Python döngüsü\", fmt_time(t2)]])\n    R_.test(\"batched_playouts\", po)\n\n    def mc():\n        ensure(need(ns, \"mcts_move\")([1, 1, 0, -1, -1, 0, 0, 0, 0], 1, 100, 8, torch.Generator().manual_seed(0)) == 2, \"kazanan hamleyi bulmalı\")\n        ensure(ns[\"mcts_move\"]([1, 1, 0, -1, 0, 0, 0, 0, 0], -1, 200, 8, torch.Generator().manual_seed(0)) == 2, \"rakibin kazanmasını engellemeli\")\n        rows = []\n        r1 = [play(\"mcts\", \"random\", s, ns) for s in range(10)]\n        r2 = [play(\"random\", \"mcts\", s, ns) for s in range(10)]\n        r3 = [play(\"mcts\", \"perfect\", s, ns) for s in range(4)] + [-play(\"perfect\", \"mcts\", s, ns) for s in range(4)]\n        ensure(all(v == 1 for v in r1) and all(v <= 0 for v in r2), \"rastgele oyuncuya karşı kaybetmemeli\")\n        ensure(all(v >= 0 for v in r3), \"kusursuz oyuncuya karşı kaybetmemeli (beraberlik)\")\n        rows.append([\"MCTS (X) – rastgele\", f\"{r1.count(1)}G {r1.count(0)}B {r1.count(-1)}M\"])\n        rows.append([\"rastgele – MCTS (O)\", f\"{r2.count(-1)}G {r2.count(0)}B {r2.count(1)}M (MCTS bakışından)\"])\n        rows.append([\"MCTS – kusursuz (her iki renk)\", f\"{r3.count(1)}G {r3.count(0)}B {r3.count(-1)}M\"])\n        R_.table(\"UCT, hamle başına 200 benzetim × 16 paralel oyun\", [\"eşleşme\", \"sonuç\"], rows,\n                 note=\"Tic-tac-toe kusursuz oyunda beraberedir. MCTS hiçbir değerlendirme fonksiyonu bilmeden yalnız rastgele oyunların istatistikleriyle bu seviyeye ulaşır.\")\n    R_.test(\"mcts_move\", mc)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        res = {}\n        for plus in (False, True):\n            R = [need(ns, \"run\")(plus, 6000, 3000, 1e-3, s) for s in range(5)]\n            res[plus] = [sum(r[k] for r in R) / 5 for k in (999, 2999, 3999, 4999, 5999)]\n        slope = {p: (v[-1] - v[1]) / 3000 for p, v in res.items()}\n        ensure(slope[True] > 1.25 * slope[False], f\"Dyna-Q+ kısayolu kullanmalı (eğimler {slope})\")\n        R_.table(\"Kısayol labirenti, kümülatif ödül (5 deney); 3000. adımda kısayol açılır\", [\"yöntem\", \"1000\", \"3000\", \"4000\", \"5000\", \"6000\", \"sonrası adım/bölüm\"],\n                 [[\"Dyna-Q\" if not p else \"Dyna-Q+\"] + [f\"{x:.0f}\" for x in v] + [f\"{1 / slope[p]:.1f}\"] for p, v in res.items()],\n                 note=\"Dyna-Q eski (hâlâ çalışan) yolu kullanmaya devam eder; Dyna-Q+ uzun süredir denenmeyen geçişleri planlamada cazip kılar, kısayolu keşfeder (Sutton Şekil 8.5).\")\n    R_.test(\"Dyna-Q+\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        ensure(need(ns, \"greedy_len\")({}) == 999, \"boş Q yol bulmaz\")\n        rows = []\n        res = {}\n        for n in (5, 20):\n            ps = [need(ns, \"prioritized\")(n, 1e-4, s) for s in range(5)]\n            dq = [need(ns, \"dyna_updates\")(n, s) for s in range(5)]\n            res[n] = (sum(ps) / 5, sum(dq) / 5)\n            rows.append([n, f\"{res[n][0]:.0f}\", f\"{res[n][1]:.0f}\", f\"{res[n][1] / res[n][0]:.1f}×\"])\n        ensure(all(v[0] < v[1] for v in res.values()), \"öncelikli tarama daha az güncellemeyle optimal yolu bulmalı\")\n        R_.table(\"Optimal yola (14 adım) kadar Q güncellemesi, 5 tohum\", [\"n\", \"öncelikli tarama\", \"Dyna-Q (düzgün)\", \"oran\"], rows,\n                 note=\"Düzgün örnekleme güncellemelerin çoğunu değeri hiç değişmeyecek çiftlere harcar; öncelikli tarama değişimi hedeften geriye, öncüller boyunca yayar.\")\n    R_.test(\"öncelikli tarama\", t)\n",
 "x3": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        v, m, n = need(ns, \"minimax\")([0] * 9, 1)\n        ensure(v == 0 and n == 549946, f\"boş tahta: beraberlik, 549 946 düğüm (bulunan {v}, {n})\")\n        va, ma, na = need(ns, \"alphabeta\")([0] * 9, 1)\n        ensure(va == 0 and na < n / 10, \"α–β aynı değer, çok daha az düğüm\")\n        b = [1, -1, 0, 0, 1, 0, 0, 0, 0]\n        ensure(ns[\"minimax\"](b, -1)[0] == 1 and ns[\"alphabeta\"](b, -1)[0] == 1, \"O 8'i kapatsa da X kazanır (çatal)\")\n        R_.table(\"Tic-tac-toe oyun ağacı\", [\"yöntem\", \"değer\", \"ziyaret edilen düğüm\"], [[\"minimax\", v, n], [\"α–β (hamle sırası 0..8)\", va, na]],\n                 note=\"Go'da dallanma ~250, derinlik ~150: tam arama imkânsız. MCTS aramayı umut veren dallara yoğunlaştırır, AlphaGo da politika ağıyla dallanmayı, değer ağıyla derinliği kısar.\")\n    R_.test(\"minimax ve α–β\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_10_planning as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Modelsiz** (Q-öğrenme, politika gradyanı, aktör–eleştirmen) ve yalnız ödül modelli (haydutlar) yöntemlerden sonra: **açık geçiş ve ödül modeli** öğren ve onunla **planla**. Kazanç: örnek verimliliği. Bedel: karmaşıklık, model hatası.

**Döngü:** eylem seç ve uygula → durum ve ödülü gözle → modeli güncelle → değeri/politikayı güncelle. Modeller: tablo (sayımlar: $\hat P(s'|s,a)=n(s,a,s')/n(s,a)$), doğrusal ($s'\approx As+Ba$), doğrusal olmayan (sinir ağı; rastgele ya da deterministik).

**Kısmi planlama:** karmaşık modelde her adımda değeri tam optimize etmek pahalı; bunun yerine birkaç simüle Q-öğrenme (ya da politika gradyanı) adımı → **Dyna-Q**. Tekrar tamponuna çok benzer: tampon basit ve gerçek örnekler, genelleme yok; model karmaşık, simüle örnekler, başka (s,a)'lara genelleyebilir.

**Şimdiki durumdan planla:** rastgele $\hat s$'lerde değil, şu anki durumdan ileri bak — bir sonraki eylemi iyileştirir. Sürekli kontrolde **MPC** (model öngörülü kontrol: birçok eylem dizisini modelde dene, en iyisinin ilk eylemini uygula, tekrarla). Ayrık karar problemlerinde **Monte Carlo ağaç araması** üç fikri birleştirir:

- yaprak düğümleri: değeri varsayılan politikayla (rastgele oyun) kestir;
- şans düğümleri: beklentiyi modelden örnekleyerek yaklaşıkla;
- karar düğümleri: yalnız umut veren eylemleri genişlet.

Dört adım: **seçim** (UCT: $\arg\max_a\bar Q(s,a)+c\sqrt{\ln N(s)/n(s,a)}$), **genişletme**, **simülasyon**, **geri yayılım**. AlphaGo'nun varyantında her kenar $Q(s,a)$, $\pi(a|s)$ (politika ağının önseli) ve $n(s,a)$ saklar; $a^*=\arg\max_aQ(s,a)+u(s,a)$, $u\propto\pi(a|s)/(1+n(s,a))$.

## 1 · Elle

UCT: $\sqrt2\sqrt{\ln100/10}=\sqrt{0.92103}=0.95971$ → 1.45971. PUCT: $\sqrt{100}=10$, $0.3\cdot10/5=0.6$.

Rastgele tic-tac-toe: X %58.5, O %28.8, beraberlik %12.7 → $0.585-0.288=0.297$ (e aşamasında 10⁵ rastgele oyunla ölçülüyor).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Ağaç araması seçim puanları**

- `uct`: UCT: $\bar Q=0.5$, $N=100$, $n=10$, $c=\sqrt2$
- `puct`: PUCT: $Q=0.2$, $\pi=0.3$, $N=100$, $n=4$, $c=1$

**Model ve Dyna**

- `ph`: (s,a)'dan s₁ 3 kez, s₂ 1 kez gözlendi: $\hat P(s_1|s,a)$
- `up`: Dyna-Q, n = 5, 50 gerçek adım: toplam Q güncellemesi
- `kp`: Dyna-Q+ bonusu $\kappa\sqrt\tau$, κ = 0.01, τ = 100

**Tic-tac-toe, boş tahtadan rastgele oyunlar**

- `tt`: X'in bakışından beklenen sonuç (X kazanır %58.5, O %28.8)

In [ ]:
cevaplar = dict(
    uct=None,
    puct=None,
    ph=None,
    up=None,
    kp=None,
    tt=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"uct":[1.4597051824376164,0.000001],"puct":[0.8,0],"ph":[0.75,0],"up":[300,0],"kp":[0.1,0],"tt":[0.297,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**Dyna labirenti** (Sutton Şekil 8.2), saf Python. 6×9, duvarlar `{(1,2),(2,2),(3,2),(4,5),(0,7),(1,7),(2,7)}`, başlangıç (2,0), hedef (0,8) (ödül 1, bölüm biter). Eylemler yukarı, aşağı, sol, sağ; duvar/kenar → yerinde kal. γ = 0.95, α = 0.1, ε = 0.1, eşitlikler rastgele.

- `maze_step(s, a)` → `(s2, r, done)`.
- `dyna_q(n_eps, n_plan, seed=0)` → `(Q, bölüm başına adım listesi)`: her gerçek adımdan sonra Q-öğrenme, model[(s,a)] = (r, s′, done), sonra `n_plan` kez modelden rastgele **görülmüş** bir (s,a) seç ve simüle Q-öğrenme adımı.

In [ ]:
%%writefile stage_b.py
import random

H, W = 6, 9
WALLS = {(1, 2), (2, 2), (3, 2), (4, 5), (0, 7), (1, 7), (2, 7)}
START, GOAL = (2, 0), (0, 8)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def maze_step(s, a):
    ...


def dyna_q(n_eps, n_plan, seed=0, alpha=0.1, gamma=0.95, eps=0.1):
    ...


print(dyna_q(20, 5)[1])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Kesinlik eşdeğerli planlama**, NumPy. Rastgele bir MDP'den (`P (A,S,S)`, `R (S,A)` belirli) düzgün rastgele politikayla toplanan geçişler `(S_, A_, R_, S2_)`.

- `fit_model(S_, A_, R_, S2_, nS, nA)` → `(P_hat, R_hat)`: sayımlar (`np.add.at`); görülmemiş (s,a) için kendine geçiş ve ödül 0.
- `plan(P, R, gamma, n_iter=500)` → Q (değer yinelemesi, yalnız yineleme döngüsü).
- `q_learning_replay(S_, A_, R_, S2_, nS, nA, gamma, epochs=1, alpha=0.1)` — aynı veri üzerinde tablo Q-öğrenme (epoch kadar tekrar; döngü serbest).
- `policy_value(P, R, pi, gamma)` → V (doğrusal sistem).

In [ ]:
%%writefile stage_c.py
import numpy as np


def fit_model(S_, A_, R_, S2_, nS, nA):
    ...


def plan(P, R, gamma, n_iter=500):
    ...


def q_learning_replay(S_, A_, R_, S2_, nS, nA, gamma, epochs=1, alpha=0.1):
    ...


def policy_value(P, R, pi, gamma):
    ...


rng = np.random.default_rng(0)
P = rng.random((2, 5, 5)); P /= P.sum(-1, keepdims=True); R = rng.random((5, 2))
S_ = rng.integers(0, 5, 100); A_ = rng.integers(0, 2, 100)
S2_ = np.array([rng.choice(5, p=P[a, s]) for s, a in zip(S_, A_)]); R_ = R[S_, A_]
Ph, Rh = fit_model(S_, A_, R_, S2_, 5, 2)
print(np.abs(Ph - P).max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**Öğrenilmiş sinir ağı modeliyle MPC** (Nagabandi ve ark. 2018), CartPole (gerçek fizik `cartpole_step` verildi, modelden gizli).

- `random_data(n_eps, g)` → `(X, A, X2)`: rastgele politika, bölüm en fazla 200.
- `class Model(nn.Module)` — 5 → 128 → 128 → 4 (ReLU), **fark** tahmini: $\hat s'=s+\text{net}((s,a)\text{ normalize})\cdot\sigma_\Delta$; normalizasyon istatistikleri tampon (buffer).
- `fit(model, X, A, X2, epochs=100, seed=0)` — normalize Δ üzerinde MSE, Adam 1e-3, batch 256; son kayıp.
- `mpc_action(model, x, g, n_seq=500, H=20)` — **rastgele atış**: 500 rastgele ikili eylem dizisini modelde paralel ilerlet; maliyet $\sum_h[10\cdot\mathbb 1(\text{düştü})+\theta_h^2+0.01x_h^2]$ (düşüş, bir kez düştükten sonra kalıcı); en iyi dizinin ilk eylemi.
- `run_mpc(model, g, max_len=200)` → bölüm uzunluğu.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn

G_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02


def cartpole_step(x, a):
    pos, vel, th, om = x.unbind(-1)
    f = torch.where(a == 1, FM, -FM).to(x.dtype)
    ct, st = torch.cos(th), torch.sin(th)
    tm, pml = MC + MP, MP * L
    temp = (f + pml * om ** 2 * st) / tm
    thacc = (G_ * st - ct * temp) / (L * (4.0 / 3.0 - MP * ct ** 2 / tm))
    xacc = temp - pml * thacc * ct / tm
    x2 = torch.stack([pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc], -1)
    done = (x2[..., 0].abs() > 2.4) | (x2[..., 2].abs() > 12 * 2 * math.pi / 360)
    return x2, done


def random_data(n_eps, g):
    ...


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x, a):
        ...


def fit(model, X, A, X2, epochs=100, seed=0):
    ...


def mpc_action(model, x, g, n_seq=500, H=20):
    ...


def run_mpc(model, g, max_len=200):
    ...


g = torch.Generator().manual_seed(0)
X, A, X2 = random_data(50, g)
m = Model(); fit(m, X, A, X2, 20)
print(run_mpc(m, g, 50))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**MCTS + vektörel rastgele oyunlar**, tic-tac-toe. Tahta 9 elemanlı liste (1 = X, −1 = O, 0 = boş).

- `winner(B)` — `B (N, 9)` tensör → `(N,)` 1/−1/0 (8 çizgi indeks tensörüyle, döngüsüz).
- `batched_playouts(board, player, n, g)` — sırası `player`'da olan tahtadan **n rastgele oyunu aynı anda** oyna (her hamlede `torch.multinomial` ile boş karelerden biri; biten oyunlar donar) → `(n,)` sonuç (X'in bakışından). Hamle döngüsü (≤ 9) serbest, oyun döngüsü yok.
- `mcts_move(board, player, n_sims, n_rollouts, g, c=1.4)` — UCT: seçim (sırası gelen oyuncunun bakışından $\bar Q\cdot\text{oyuncu}+c\sqrt{\ln N/n}$), genişletme (bir çocuk), simülasyon (`n_rollouts` paralel oyunun ortalaması ya da terminal değer), geri yayılım (X bakışından değer); en çok ziyaret edilen hamle.

In [ ]:
%%writefile stage_e.py
import math
import torch

LINES = torch.tensor([[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]])


def winner(B):
    ...


def batched_playouts(board, player, n, g):
    ...


def mcts_move(board, player, n_sims, n_rollouts, g, c=1.4):
    ...


g = torch.Generator().manual_seed(0)
print(batched_playouts([0] * 9, 1, 10000, g).mean())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Değişen ortam: Dyna-Q+ ile kısayol labirenti

**Kısayol labirenti** (Sutton Örnek 8.3): 6×9, başlangıç (5,3), hedef (0,8). 3. satırda duvar: başta sütun 1–8 (geçit yalnız solda), 3000. adımda sütun 8 açılır (sağda kısayol). α = 1, ε = 0.1, γ = 0.95, n = 50.

- `run(plus, n_steps=6000, change=3000, kappa=1e-3, seed=0)` → kümülatif ödül listesi. `plus=True` (Dyna-Q+): planlamada ödüle $\kappa\sqrt{\tau}$ ekle (τ: (s,a)'nın son gerçek denenmesinden beri geçen adım); ilk kez görülen durumda denenmemiş eylemler modele "aynı durumda kal, ödül 0" olarak eklenir.

In [ ]:
%%writefile extra_x1.py
import math
import random

H, W = 6, 9
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
START, GOAL = (5, 3), (0, 8)
WALL_A = {(3, c) for c in range(1, 9)}
WALL_B = {(3, c) for c in range(1, 8)}


def run(plus, n_steps=6000, change=3000, kappa=1e-3, seed=0, n_plan=50):
    ...


print(run(True, 2000)[-1])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Öncelikli tarama

**Öncelikli tarama** (Moore & Atkeson 1993) — Dyna labirentinde (b ile aynı, deterministik model).

- `prioritized(n_plan, theta=1e-4, seed=0, max_steps=20000)` → optimal yolu (14 adım) açgözlü politika bulana dek yapılan **toplam Q güncellemesi**; gerçek adımın önceliği $|\delta|$, θ'dan büyükse kuyruğa (`heapq`, negatif öncelik); planlamada en öncelikli çifti güncelle, sonra ona **önceden giden** (s̄, ā) çiftlerinin önceliklerini hesapla.
- `dyna_updates(n_plan, seed=0, max_steps=20000)` — aynı ölçü, düz Dyna-Q.

In [ ]:
%%writefile extra_x2.py
import heapq
import random

H, W = 6, 9
WALLS = {(1, 2), (2, 2), (3, 2), (4, 5), (0, 7), (1, 7), (2, 7)}
START, GOAL = (2, 0), (0, 8)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def maze_step(s, a):
    r, c = s[0] + MOVES[a][0], s[1] + MOVES[a][1]
    if not (0 <= r < H and 0 <= c < W) or (r, c) in WALLS:
        r, c = s
    return (r, c), (1.0 if (r, c) == GOAL else 0.0), (r, c) == GOAL


def greedy_len(Q):
    s = START
    for k in range(1, 60):
        qs = Q.get(s, [0.0] * 4)
        s, _, done = maze_step(s, qs.index(max(qs)))
        if done:
            return k
    return 999


def prioritized(n_plan, theta=1e-4, seed=0, max_steps=20000):
    ...


def dyna_updates(n_plan, seed=0, max_steps=20000):
    ...


print(prioritized(5), dyna_updates(5))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Minimax ve alfa–beta budama

- `minimax(board, player)` → `(değer, hamle, ziyaret edilen düğüm)`; `alphabeta(board, player)` aynı, α–β budamasıyla (hamle sırası 0..8). Tahta b/e ile aynı gösterim.

In [ ]:
%%writefile extra_x3.py
def minimax(board, player):
    ...


def alphabeta(board, player):
    ...


print(minimax([0] * 9, 1)[::2], alphabeta([0] * 9, 1)[::2])

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Kocsis & Szepesvári (2006): UCT

[Kocsis & Szepesvári — Bandit Based Monte-Carlo Planning (ECML 2006)](https://doi.org/10.1007/11871842_29)

UCT'yi makalenin P-game ağaçlarında (rastgele dallanma 2–8, derinlik 4–20, yapraklarda rastgele değer) uygula ve minimax değerine göre hata–örnek sayısı eğrisini, alfa–beta ve düz Monte Carlo (UCB'siz) ile karşılaştır. Sonra d/e aşamalarındaki tic-tac-toe kodunu Connect-4'e (6×7) genişlet ve UCT'yi saf rastgele oyuncuya ve sınırlı derinlikte minimax'a karşı oynat.


**Adımlar**

1. e aşamasındaki vektörel rastgele oyun fonksiyonunu Connect-4 kazanma çizgileriyle genelleştir.
2. Benzetim sayısı 100, 1000, 10 000.

**Kabul ölçütleri**

- UCT'nin hatası örnekle düz MC'den hızlı düşer (makale Şekil 2 nitel).
- Connect-4'te 10 000 benzetimli UCT, rastgele oyuncuyu ≥ %95 yener.

## Makale ödevi: Nagabandi ve ark. (2018): Sinir ağı dinamik modeli + MPC

[Nagabandi ve ark. — Neural Network Dynamics for Model-Based Deep RL with Model-Free Fine-Tuning (ICRA 2018)](https://arxiv.org/abs/1708.02596)

d aşamasındaki modeli Gymnasium MuJoCo (HalfCheetah, Swimmer) ortamlarına taşı: girdi/çıktı normalizasyonu, Δs tahmini, rastgele atış MPC (K = 1000, H = 10–20), veri birleştirme (Algoritma 1); ardından MPC'yi taklit eden bir politikayla model-free (PPO) ince ayarı başlat (Bölüm IV-B).


**Adımlar**

1. Çok adımlı model hatasını H'ye karşı ölç.
2. Örnek sayısına göre ödül eğrisi: MB, MB-MF ve saf MF.

**Kabul ölçütleri**

- MB, MF'ye göre 1–2 büyüklük mertebesi az örnekle makul performansa ulaşır (makale Şekil 6 nitel).

## Kaynaklar

- [Sutton — Dyna, an Integrated Architecture for Learning, Planning, and Reacting (SIGART 1991)](https://doi.org/10.1145/122344.122377) — ©
- [Kocsis & Szepesvári — Bandit Based Monte-Carlo Planning (ECML 2006)](https://doi.org/10.1007/11871842_29) — ©
- [Browne ve ark. — A Survey of Monte Carlo Tree Search Methods (IEEE TCIAIG 2012)](https://doi.org/10.1109/TCIAIG.2012.2186810) — ©
- [Nagabandi ve ark. — Neural Network Dynamics for Model-Based Deep RL with Model-Free Fine-Tuning (ICRA 2018)](https://arxiv.org/abs/1708.02596) — arXiv
- [Moore & Atkeson — Prioritized Sweeping (Machine Learning 1993)](https://doi.org/10.1007/BF00993104) — ©
- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 8](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0